# 5. Abstract Classes

Abstract classes define a **contract** that subclasses must fulfill — same concept as Java's `abstract class`. Python uses the `abc` module (Abstract Base Classes) to enforce this.

This notebook covers: `ABC`, `@abstractmethod`, abstract properties, abstract methods with a body, concrete methods in abstract classes, and when to use ABC vs Protocol.

### 5.1 Defining an Abstract Class

**☕ JAVA:**
```java
public abstract class Shape {
    public abstract double area();
    public abstract double perimeter();

    // Concrete method in abstract class
    public String describe() {
        return getClass().getSimpleName() + ": area=" + area();
    }
}
```

**🐍 PYTHON:** Inherit from `ABC` and use `@abstractmethod`. You can also have concrete (non-abstract) methods in the same class.

In [15]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass
    
    @abstractmethod
    def perimeter(self) -> float:
        ... # ellipsis

    def describe(self) -> str:
        return f"{self.__class__.__name__}: area= {self.area():.2f}, perimeter= {self.perimeter():0.2f}"

In [16]:
try:
    s = Shape()
except TypeError as e:
    print(f"Error {e}")

Error Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'


### 5.2 Implementing an Abstract Class

**☕ JAVA:** Subclass uses `extends` and must implement all abstract methods, or be abstract itself.

**🐍 PYTHON:** Same rule — all `@abstractmethod` methods must be implemented, or `TypeError` at instantiation.

In [9]:
class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height
    
    def area(self) -> float:
        return self.width * self.height
    
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius        
    
    def area(self) -> float:
        return math.pi * (self.radius ** 2)
    
    def perimeter(self) -> float:
        return math.pi * 2 * self.radius 

shapes: list[Shape] = [Rectangle(5, 7), Circle(7)]

for shape in shapes:
    print(f"{shape.describe()}")

Rectangle: area= 35.00, perimeter= 24.00
Circle: area= 153.94, perimeter= 43.98


In [17]:
class IncompleteShape(Shape):
    def area(self):
        return 0
    
    # perimeter() not implemented

try:
    s = IncompleteShape()
    print(s.area())
except TypeError as e:
    print(f"Error {e}")

Error Can't instantiate abstract class IncompleteShape without an implementation for abstract method 'perimeter'


### 5.3 Abstract Properties

**☕ JAVA:** Abstract getters are just abstract methods: `public abstract String getName();`

**🐍 PYTHON:** Combine `@property` with `@abstractmethod` — the subclass must implement it as a property.

In [ ]:
@property
@abstractmethod
def area(self):
    pass

  Dog: Woof!, 4 legs
  Snake: Hiss!, 0 legs


### 5.4 Abstract Methods with a Body — `super()` Trick

**☕ JAVA:** Abstract methods have **no body** — `public abstract double area();`

**🐍 PYTHON:** Abstract methods **can have a body**! Subclasses can call it via `super()` for shared default behavior. This is a powerful pattern for providing a base implementation that subclasses extend.

> 💡 **Key insight:** This is a Java/Python difference that surprises many developers. In Java, abstract methods cannot have a body at all. In Python, the `@abstractmethod` only enforces that subclasses *define* the method — the abstract method's body is available via `super()`.

### 5.5 `__init__` in Abstract Classes — Common Gotcha

**☕ JAVA:** Abstract classes commonly have constructors that subclasses call via `super()`.

**🐍 PYTHON:** Same pattern works! But note: **don't make `__init__` abstract**. Instead, define a concrete `__init__` in the ABC and have subclasses call `super().__init__()`.

User(id=1): valid=True, created=16:57:17
Product(id=2): valid=True, created=16:57:17


> ⚠️ **Why not `@abstractmethod` on `__init__`?** Technically possible, but pointless — every class already needs `__init__` to be useful. The ABC is better off providing a concrete `__init__` with shared setup, and letting subclasses extend it via `super().__init__()`.

### 5.6 `ABC` vs `ABCMeta` — Under the Hood

`ABC` is just a convenience class. Under the hood, it uses `ABCMeta` as its metaclass:

```python
class ABC(metaclass=ABCMeta):   # This is all ABC does!
    pass
```

You only need `ABCMeta` directly when your class already has a different metaclass (rare).

> 💡 **Rule of thumb:** Always use `class MyABC(ABC):` unless you get a metaclass conflict error — then switch to `class MyABC(OtherBase, metaclass=ABCMeta):`.

### 5.7 ABC with `__init_subclass__` — Registration Pattern

**☕ JAVA:** No direct equivalent — you'd use a registry `Map<String, Class>`.

**🐍 PYTHON:** `__init_subclass__` is called automatically when a class is subclassed. Great for plugin systems or automatic registration.

### 5.8 ABC vs Protocol — When to Use Which?

| Feature | ABC | Protocol |
|---------|-----|----------|
| Style | Nominal (must inherit) | Structural (duck typing) |
| Java equivalent | `abstract class` / `interface` | No equivalent |
| Must inherit? | ✅ Yes | ❌ No — just implement the methods |
| Runtime check | `isinstance()` works | Only with `@runtime_checkable` |
| Enforcement | At instantiation time | At type-checking time (mypy) |
| Best for | Frameworks, plugins | Loose coupling, libraries |

  Drawing a square


---

## 🧪 Try It Yourself

**Exercise 1:** Create an abstract `Database` class with abstract methods `connect()`, `disconnect()`, and `query(sql)`. Implement `SQLiteDB` and `PostgresDB` that print simulated behavior.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create an abstract `Validator` with an abstract property `pattern` and a concrete method `validate(text)` that checks if text matches the pattern using `re.match()`. Implement `EmailValidator` and `PhoneValidator`.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create an abstract `PaymentProcessor` with `__init_subclass__` auto-registration. Implement `CreditCardProcessor` and `PayPalProcessor`. Write a factory function that creates a processor by name from the registry.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Abstract class | `abstract class Shape` | `class Shape(ABC):` |
| Abstract method | `public abstract double area();` | `@abstractmethod` |
| Import | Built-in keyword | `from abc import ABC, abstractmethod` |
| Abstract method body | ❌ Not allowed | ✅ Can have body, callable via `super()` |
| Concrete method in ABC | ✅ Same | ✅ Same |
| Abstract property | Abstract getter | `@property` + `@abstractmethod` |
| Abstract `__init__` | Common pattern | ❌ Don't — use concrete `__init__` instead |
| `ABC` vs `ABCMeta` | N/A | `ABC` = convenience; `ABCMeta` = for metaclass conflicts |
| Instantiation check | Compile-time error | Runtime `TypeError` |
| Missing method check | Compile-time error | Runtime `TypeError` at instantiation |
| Plugin registry | Manual `Map<String, Class>` | `__init_subclass__` auto-registration |
| Duck typing interface | Not possible | `Protocol` (structural typing) |
| Best practice | Use `abstract class` or `interface` | Prefer Protocol unless you need enforcement |